# Stage 3-3. Losses and Metrics

이 노트북은 `src/nn/losses.py`와 `src/nn/metrics.py`에 구현된 손실 함수, gradient 함수, 평가 지표를 실습한다.

| 함수 | task | 입력 | 출력 |
|---|---|---|---|
| `cross_entropy` | multiclass | logits `(N, C)`, one-hot targets `(N, C)` | scalar |
| `cross_entropy_grad` | multiclass | 위와 동일 | `(N, C)` |
| `binary_cross_entropy` | binary | logits `(N, 1)`, binary targets `(N, 1)` | scalar |
| `binary_cross_entropy_grad` | binary | 위와 동일 | `(N, 1)` |
| `mse` | regression | preds `(N, 1)`, targets `(N, 1)` | scalar |
| `mse_grad` | regression | 위와 동일 | `(N, 1)` |
| `accuracy` | multiclass | logits `(N, C)`, one-hot targets `(N, C)` | scalar |
| `binary_accuracy` | binary | logits `(N, 1)`, binary targets `(N, 1)` | scalar |
| `r2_score` | regression | preds `(N, 1)`, targets `(N, 1)` | scalar |

**학습 목표**
1. 3종 loss 함수의 값과 gradient를 계산하고 수식과 비교한다.
2. loss 내부에서 activation(softmax, sigmoid)이 처리되는 이유를 이해한다.
3. 3종 metric 함수의 동작을 확인한다.
4. loss 값이 학습 과정에서 어떻게 변화하는지 시뮬레이션한다.

## 0. 환경 설정

In [ ]:
import sys
sys.path.insert(0, "../..")

import numpy as np
import matplotlib.pyplot as plt

from src.nn.losses import (
    cross_entropy, cross_entropy_grad,
    binary_cross_entropy, binary_cross_entropy_grad,
    mse, mse_grad,
)
from src.nn.metrics import accuracy, binary_accuracy, r2_score
from src.nn.activations import softmax, sigmoid

## 1. Cross-Entropy Loss — multiclass

In [ ]:
# 샘플 4개, 클래스 3개
rng = np.random.default_rng(42)
logits_mc = rng.normal(0, 1, (4, 3)).astype(np.float32)

# one-hot targets
labels_mc = np.array([0, 2, 1, 0])
targets_mc = np.eye(3, dtype=np.float32)[labels_mc]  # (4, 3)

loss_mc = cross_entropy(logits_mc, targets_mc)
grad_mc = cross_entropy_grad(logits_mc, targets_mc)

print("=== Cross-Entropy (multiclass) ===")
print(f"logits shape : {logits_mc.shape}")
print(f"targets shape: {targets_mc.shape}")
print(f"loss         : {loss_mc:.4f}")
print(f"grad shape   : {grad_mc.shape}")
print(f"grad (첫 샘플): {grad_mc[0].round(4)}")
print()
print("수식 검증: grad = (softmax(logits) - targets) / N")
manual_grad_mc = (softmax(logits_mc) - targets_mc) / len(logits_mc)
print(f"일치 여부: {np.allclose(grad_mc, manual_grad_mc, atol=1e-6)}")

> **설계 원칙**: `cross_entropy`는 내부에서 `softmax`를 적용한 뒤 NLL을 계산한다.  
> gradient 역시 `softmax(logits) - targets`로 단순화된다 — softmax와 cross-entropy를 합산한 결합 gradient 공식이다.

## 2. Binary Cross-Entropy Loss — binary

In [ ]:
logits_bin = rng.normal(0, 1, (4, 1)).astype(np.float32)
targets_bin = np.array([[1.], [0.], [1.], [0.]], dtype=np.float32)

loss_bin = binary_cross_entropy(logits_bin, targets_bin)
grad_bin = binary_cross_entropy_grad(logits_bin, targets_bin)

print("=== Binary Cross-Entropy ===")
print(f"logits  : {logits_bin.T}")
print(f"targets : {targets_bin.T}")
print(f"loss    : {loss_bin:.4f}")
print(f"grad    : {grad_bin.T.round(4)}")
print()
print("수식 검증: grad = (sigmoid(logits) - targets) / N")
manual_grad_bin = (sigmoid(logits_bin) - targets_bin) / len(logits_bin)
print(f"일치 여부: {np.allclose(grad_bin, manual_grad_bin, atol=1e-6)}")

## 3. MSE Loss — regression

In [ ]:
preds_reg = rng.normal(0, 1, (4, 1)).astype(np.float32)
targets_reg = np.array([[0.1], [0.5], [0.9], [0.3]], dtype=np.float32)

loss_reg = mse(preds_reg, targets_reg)
grad_reg = mse_grad(preds_reg, targets_reg)

print("=== MSE ===")
print(f"preds   : {preds_reg.T.round(4)}")
print(f"targets : {targets_reg.T}")
print(f"loss    : {loss_reg:.4f}")
print(f"grad    : {grad_reg.T.round(4)}")
print()
print("수식 검증: grad = 2*(preds - targets) / N")
manual_grad_reg = 2.0 * (preds_reg - targets_reg) / len(preds_reg)
print(f"일치 여부: {np.allclose(grad_reg, manual_grad_reg, atol=1e-6)}")

## 4. 3종 Loss 함수 비교 — 정답 확률에 따른 값 변화

In [ ]:
# 정답 클래스 확률(또는 예측값)을 변화시키며 loss 추적
prob_range = np.linspace(0.01, 0.99, 200)

# Cross-Entropy: 정답 클래스 확률이 p일 때 loss = -log(p)
ce_loss = -np.log(prob_range)

# Binary CE: 정답=1 일 때 loss = -log(p)
bce_loss = -(np.log(prob_range))

# MSE: 정답=1 이고 예측=p 일 때 loss = (1-p)^2
mse_loss = (1 - prob_range) ** 2

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(prob_range, ce_loss,  label="Cross-Entropy: −log(p)",   color="steelblue",     linewidth=2)
ax.plot(prob_range, bce_loss, label="BCE: −log(p)",             color="#E87B4C",       linewidth=2, linestyle="--")
ax.plot(prob_range, mse_loss, label="MSE: (1−p)²",              color="mediumseagreen",linewidth=2)

ax.set_title("Loss vs 정답 클래스 예측 확률", fontsize=12)
ax.set_xlabel("정답 클래스 예측 확률 p")
ax.set_ylabel("loss")
ax.legend()
ax.set_xlim(0, 1)
ax.set_ylim(0, 5)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

> **관찰**: Cross-Entropy는 예측이 틀렸을 때(`p → 0`) loss가 급격히 증가해 강한 gradient를 만든다.  
> MSE는 확률 스케일에서 완만하여 분류 문제에 덜 효과적이다.

## 5. Metrics — accuracy (multiclass)

In [ ]:
# logits에서 argmax로 예측 → one-hot targets의 argmax와 비교
logits_acc = np.array([
    [2.0, 0.5, 0.1],   # 예측: 0, 정답: 0  ✓
    [0.1, 0.8, 3.0],   # 예측: 2, 정답: 2  ✓
    [1.0, 2.0, 0.5],   # 예측: 1, 정답: 0  ✗
    [0.2, 0.1, 0.9],   # 예측: 2, 정답: 1  ✗
], dtype=np.float32)

targets_acc = np.eye(3, dtype=np.float32)[[0, 2, 0, 1]]

acc = accuracy(logits_acc, targets_acc)

print("=== accuracy (multiclass) ===")
print(f"예측 클래스  : {logits_acc.argmax(axis=1)}")
print(f"정답 클래스  : {targets_acc.argmax(axis=1)}")
print(f"accuracy     : {acc:.4f}  (2/4 = 0.5 기대)")

## 6. Metrics — binary_accuracy

In [ ]:
# logits >= 0 이면 예측 1, 아니면 0 (sigmoid(x) >= 0.5 와 동치)
logits_bacc = np.array([[1.5], [-0.3], [0.0], [-2.0]], dtype=np.float32)
targets_bacc = np.array([[1.], [0.], [1.], [0.]], dtype=np.float32)

bacc = binary_accuracy(logits_bacc, targets_bacc)

preds_label = (logits_bacc >= 0.0).astype(int).flatten()
true_label  = targets_bacc.astype(int).flatten()

print("=== binary_accuracy ===")
print(f"logits  : {logits_bacc.T}")
print(f"예측     : {preds_label}")
print(f"정답     : {true_label}")
print(f"일치     : {preds_label == true_label}")
print(f"accuracy : {bacc:.4f}  (3/4 = 0.75 기대)")

## 7. Metrics — r2_score (regression)

In [ ]:
# 완벽한 예측 vs 평균만 예측 vs 나쁜 예측
targets_r2 = np.array([[0.1], [0.5], [0.9], [0.3], [0.7]], dtype=np.float32)

perfect_preds = targets_r2.copy()
mean_preds    = np.full_like(targets_r2, targets_r2.mean())
bad_preds     = 1.0 - targets_r2  # 반대 방향

print("=== r2_score ===")
print(f"완벽한 예측 r²: {r2_score(perfect_preds, targets_r2):.4f}  (기대: ~1.0)")
print(f"평균값 예측  r²: {r2_score(mean_preds, targets_r2):.4f}  (기대: ~0.0)")
print(f"나쁜 예측    r²: {r2_score(bad_preds,  targets_r2):.4f}  (기대: 음수)")

> **r² 해석**: 1에 가까울수록 좋음. 0은 평균 예측 수준, 음수는 평균보다 나쁜 예측.

## 8. Loss / Metric 학습 시뮬레이션

In [ ]:
# 단일 Linear 레이어로 3개 task의 loss 하강 과정 시뮬레이션
from src.nn.layers import Linear
from src.task import get_task_spec, transform_targets

rng2 = np.random.default_rng(0)
N, IN = 64, 16

# 공통 입력
x_sim = rng2.normal(0, 1, (N, IN)).astype(np.float32)
labels_sim = rng2.integers(0, 10, N).astype(np.uint8)

tasks = [
    ("multiclass", cross_entropy,        cross_entropy_grad,        accuracy,        "Accuracy"),
    ("binary",     binary_cross_entropy, binary_cross_entropy_grad, binary_accuracy, "Binary Acc"),
    ("regression", mse,                  mse_grad,                  r2_score,        "R²"),
]

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
fig.suptitle("3 Task Loss / Metric (Simulated Training)", fontsize=13, fontweight="bold")

for col, (task_name, loss_fn, grad_fn, metric_fn, metric_label) in enumerate(tasks):
    spec = get_task_spec(task_name)
    targets_sim = transform_targets(labels_sim, task_name)

    layer_sim = Linear(IN, spec["output_dim"], seed=col)
    lr = 0.05
    losses, metrics = [], []

    for _ in range(60):
        out = layer_sim.forward(x_sim)
        l = loss_fn(out, targets_sim)
        m = metric_fn(out, targets_sim)
        losses.append(float(l))
        metrics.append(float(m))

        grad = grad_fn(out, targets_sim)
        layer_sim.backward(grad)
        layer_sim.w -= lr * layer_sim.grad_w
        layer_sim.b -= lr * layer_sim.grad_b

    axes[0, col].plot(losses, color="steelblue", linewidth=2)
    axes[0, col].set_title(f"{task_name} — Loss")
    axes[0, col].set_xlabel("iteration")
    axes[0, col].set_ylabel("loss")
    axes[0, col].grid(True, alpha=0.3)

    axes[1, col].plot(metrics, color="#E87B4C", linewidth=2)
    axes[1, col].set_title(f"{task_name} — {metric_label}")
    axes[1, col].set_xlabel("iteration")
    axes[1, col].set_ylabel(metric_label)
    axes[1, col].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. 정리

| 함수 | task | 입력 activation | gradient 형태 |
|---|---|---|---|
| `cross_entropy` | multiclass | softmax (내부) | `(softmax(logits) − targets) / N` |
| `binary_cross_entropy` | binary | sigmoid (내부) | `(sigmoid(logits) − targets) / N` |
| `mse` | regression | identity | `2(preds − targets) / N` |

| metric | task | 입력 | 판단 기준 |
|---|---|---|---|
| `accuracy` | multiclass | logits | `argmax(logits) == argmax(targets)` |
| `binary_accuracy` | binary | logits | `logits >= 0` |
| `r2_score` | regression | preds | `1 − SS_res / SS_tot` |

**핵심 설계 원칙**
- `cross_entropy`, `binary_cross_entropy`는 내부에서 activation을 처리하여 수치 안정성과 gradient 단순화를 달성한다.
- `metrics`는 logit 입력을 기준으로 설계되어 `trainer`/`evaluator`에서 별도 activation 호출 없이 사용한다.